In [1]:
# CLT_interactive_colab.py
# Pegar todo esto en una celda de Google Colab y ejecutar.
# Requiere: numpy, scipy, matplotlib, ipywidgets (Colab ya suele tenerlos).
# Si ipywidgets no está instalado, ejecuta: !pip install ipywidgets

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output

np.random.seed(12345)

# -------------------------
# Generador de muestras por distribución
# -------------------------
def sample_from(dist_name, n, params):
    if dist_name == 'uniform':
        a, b = params.get('a', 0.0), params.get('b', 1.0)
        return np.random.uniform(a, b, size=n)
    if dist_name == 'exponential':
        scale = params.get('scale', 1.0)
        return np.random.exponential(scale=scale, size=n)
    if dist_name == 'pareto':
        # numpy.random.pareto(a) => PDF ~ x^{-a-1} for x>1
        a = params.get('a', 1.5)
        scale = params.get('scale', 1.0)
        return (np.random.pareto(a, size=n) + 1) * scale
    if dist_name == 'cauchy':
        loc = params.get('loc', 0.0)
        scale = params.get('scale', 1.0)
        return np.random.standard_cauchy(size=n) * scale + loc
    if dist_name == 'bernoulli':
        p = params.get('p', 0.5)
        return np.random.binomial(1, p, size=n)
    if dist_name == 'poisson':
        lam = params.get('lam', 3.0)
        return np.random.poisson(lam, size=n)
    if dist_name == 'bimodal':
        mu1, mu2 = params.get('mu1', -3), params.get('mu2', 3)
        s1, s2 = params.get('sigma1', 1), params.get('sigma2', 1)
        mix = params.get('mix', 0.5)
        choices = np.random.rand(n) < mix
        arr = np.empty(n)
        arr[choices]  = np.random.normal(mu1, s1, size=choices.sum())
        arr[~choices] = np.random.normal(mu2, s2, size=(~choices).sum())
        return arr
    # por defecto normal
    return np.random.normal(0,1,size=n)

# -------------------------
# Simulación de medias muestrales
# -------------------------
def simulate_sample_means(dist_name, n, num_samples, params):
    means = np.empty(num_samples)
    for i in range(num_samples):
        x = sample_from(dist_name, n, params)
        means[i] = x.mean()
    return means

# -------------------------
# Función que dibuja y calcula métricas
# -------------------------
def analyze_and_plot(dist_name, n, num_samples, bins, show_qq,
                     # distribución: parámetros comunes (se ignorarán si no aplican)
                     a, scale, loc, p, lam, mu1, mu2, sigma1, sigma2, mix, a_pareto, b_uniform):
    # Mapear parámetros a dict
    params = {}
    if dist_name == 'uniform':
        params['a'] = b_uniform  # use b_uniform as upper bound; lower 0
        params['b'] = b_uniform
        params['a'] = 0.0
        params['b'] = b_uniform
    if dist_name == 'exponential':
        params['scale'] = scale
    if dist_name == 'pareto':
        params['a'] = a_pareto
        params['scale'] = scale
    if dist_name == 'cauchy':
        params['loc'] = loc
        params['scale'] = scale
    if dist_name == 'bernoulli':
        params['p'] = p
    if dist_name == 'poisson':
        params['lam'] = lam
    if dist_name == 'bimodal':
        params['mu1'] = mu1; params['mu2'] = mu2
        params['sigma1'] = sigma1; params['sigma2'] = sigma2
        params['mix'] = mix

    # Evitar cálculos pesados si num_samples muy grande
    if num_samples > 20000:
        print("num_samples demasiado grande (>20000). Se recomienda ≤ 20000 para interactividad.")
        return

    means = simulate_sample_means(dist_name, n, num_samples, params)

    # Estadísticas descriptivas
    mu_hat = means.mean()
    sigma_hat = means.std(ddof=1)
    skewness = stats.skew(means, bias=False)
    kurt = stats.kurtosis(means, fisher=False, bias=False)  # kurtosis de Pearson (3 para normal)

    # Test de normalidad (requiere al menos 8 muestras)
    normal_test_text = "No aplicable (num_samples < 8)"
    pvalue = None
    if num_samples >= 8:
        k2, pvalue = stats.normaltest(means)
        normal_test_text = f"D'Agostino K^2: stat={k2:.3f}, p-value={pvalue:.3e}"

    # Dibujar
    clear_output(wait=True)
    print(f"Distribución: {dist_name} — tamaño muestral n={n} — réplicas={num_samples}")
    print(f"Media de las medias: {mu_hat:.6f}  |  Desv. std (medias): {sigma_hat:.6f}")
    print(f"Skewness: {skewness:.4f} | Kurtosis (Pearson): {kurt:.4f}")
    print(normal_test_text)
    if dist_name in ('cauchy',):
        print("AVISO: Cauchy no tiene media/varianza finitas. CLT clásico NO se aplica.")

    fig, axes = plt.subplots(1, 2 if show_qq else 1, figsize=(12,4))
    if show_qq:
        ax_hist = axes[0]; ax_qq = axes[1]
    else:
        ax_hist = axes

    # Histograma de medias muestrales
    counts, bins_edges, _ = ax_hist.hist(means, bins=bins, density=True, alpha=0.6)
    xs = np.linspace(bins_edges[0], bins_edges[-1], 300)
    try:
        ax_hist.plot(xs, stats.norm.pdf(xs, mu_hat, sigma_hat), label='Normal ajustada (empírica)')
    except Exception:
        pass
    ax_hist.set_title('Histograma de medias muestrales')
    ax_hist.legend()

    if show_qq:
        stats.probplot(means, dist="norm", plot=plt)
        ax_qq = plt.gca()
        ax_qq.set_title('QQ-plot de medias muestrales')

    plt.tight_layout()
    plt.show()

    # Extra: devolver valores por si el usuario quiere inspeccionar
    return {
        'means': means,
        'mu_hat': mu_hat,
        'sigma_hat': sigma_hat,
        'skewness': skewness,
        'kurtosis': kurt,
        'normal_test_pvalue': pvalue
    }

# -------------------------
# Widgets interactivos
# -------------------------
dist_dropdown = widgets.Dropdown(
    options=[
        ('Uniform [0,b]', 'uniform'),
        ('Exponencial', 'exponential'),
        ('Pareto (cola pesada)', 'pareto'),
        ('Cauchy (sin media)', 'cauchy'),
        ('Bernoulli (0/1)', 'bernoulli'),
        ('Poisson', 'poisson'),
        ('Bimodal (mezcla gaussianas)', 'bimodal')
    ],
    value='uniform',
    description='Distribución:'
)

n_slider = widgets.IntSlider(value=30, min=1, max=2000, step=1, description='n (tamaño):')
num_samples_slider = widgets.IntSlider(value=2000, min=100, max=20000, step=100, description='réplicas:')
bins_slider = widgets.IntSlider(value=40, min=5, max=200, step=1, description='bins:')
qq_check = widgets.Checkbox(value=True, description='Mostrar QQ-plot')

# Parámetros (controles universales; se ocultarán/ignorarán según distribución)
# scale / loc / a (pareto) / p (bernoulli) / lam (poisson) / bimodal params
scale_slider = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=2, step=0.1, description='scale:')
loc_slider = widgets.FloatSlider(value=0.0, min=-10, max=10, step=0.1, description='loc:')
a_pareto_slider = widgets.FloatSlider(value=1.5, min=0.1, max=5.0, step=0.1, description='alpha (Pareto):')
p_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description='p (Bernoulli):')
lam_slider = widgets.FloatSlider(value=3.0, min=0.1, max=50.0, step=0.1, description='lambda (Poisson):')
b_uniform_slider = widgets.FloatSlider(value=1.0, min=0.1, max=20.0, step=0.1, description='b (Uniform):')

# Bimodal params
mu1_slider = widgets.FloatSlider(value=-3.0, min=-10, max=10, step=0.1, description='mu1:')
mu2_slider = widgets.FloatSlider(value=3.0, min=-10, max=10, step=0.1, description='mu2:')
sigma1_slider = widgets.FloatSlider(value=1.0, min=0.01, max=10, step=0.01, description='sigma1:')
sigma2_slider = widgets.FloatSlider(value=1.0, min=0.01, max=10, step=0.01, description='sigma2:')
mix_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description='mix:')

# Contenedor de parámetros
params_box = widgets.VBox([])

# Función para actualizar los parámetros visibles según la distribución seleccionada
def update_params_box(change=None):
    d = dist_dropdown.value
    children = []
    # añadir los widgets relevantes
    if d == 'uniform':
        children = [b_uniform_slider]
    elif d == 'exponential':
        children = [scale_slider]
    elif d == 'pareto':
        children = [a_pareto_slider, scale_slider]
    elif d == 'cauchy':
        children = [loc_slider, scale_slider]
    elif d == 'bernoulli':
        children = [p_slider]
    elif d == 'poisson':
        children = [lam_slider]
    elif d == 'bimodal':
        children = [mu1_slider, sigma1_slider, mu2_slider, sigma2_slider, mix_slider]
    params_box.children = children

# Llamar al inicio para mostrar parámetros iniciales
update_params_box()

# Observadores
dist_dropdown.observe(update_params_box, names='value')

# Botón de ejecutar (por si el usuario prefiere activarlo manualmente)
run_button = widgets.Button(description='Ejecutar simulación', button_style='success')

out = widgets.Output()

def run_button_clicked(b):
    with out:
        analyze_and_plot(
            dist_name = dist_dropdown.value,
            n = n_slider.value,
            num_samples = num_samples_slider.value,
            bins = bins_slider.value,
            show_qq = qq_check.value,
            a = None, scale = scale_slider.value, loc = loc_slider.value,
            p = p_slider.value, lam = lam_slider.value,
            mu1 = mu1_slider.value, mu2 = mu2_slider.value,
            sigma1 = sigma1_slider.value, sigma2 = sigma2_slider.value,
            mix = mix_slider.value, a_pareto = a_pareto_slider.value,
            b_uniform = b_uniform_slider.value
        )

run_button.on_click(run_button_clicked)

# Interactividad automática con observe: recalcula al cambiar cualquier control (más natural)
def auto_update(change):
    # Limitar recalculaciones demasiado frecuentes
    with out:
        analyze_and_plot(
            dist_name = dist_dropdown.value,
            n = n_slider.value,
            num_samples = num_samples_slider.value,
            bins = bins_slider.value,
            show_qq = qq_check.value,
            a = None, scale = scale_slider.value, loc = loc_slider.value,
            p = p_slider.value, lam = lam_slider.value,
            mu1 = mu1_slider.value, mu2 = mu2_slider.value,
            sigma1 = sigma1_slider.value, sigma2 = sigma2_slider.value,
            mix = mix_slider.value, a_pareto = a_pareto_slider.value,
            b_uniform = b_uniform_slider.value
        )

# Enlazar cambios (observadores) para recalcular automáticamente
for w in [dist_dropdown, n_slider, num_samples_slider, bins_slider, qq_check,
          scale_slider, loc_slider, a_pareto_slider, p_slider, lam_slider, b_uniform_slider,
          mu1_slider, mu2_slider, sigma1_slider, sigma2_slider, mix_slider]:
    w.observe(auto_update, names='value')

# Layout final
controls_left = widgets.VBox([dist_dropdown, n_slider, num_samples_slider, bins_slider, qq_check, run_button])
controls_right = widgets.VBox([params_box])
ui = widgets.HBox([controls_left, controls_right], layout=widgets.Layout(align_items='flex-start'))

display(ui, out)

# Ejecutar una vez con valores por defecto para que aparezca gráfico inicial
with out:
    analyze_and_plot(
        dist_name = dist_dropdown.value,
        n = n_slider.value,
        num_samples = num_samples_slider.value,
        bins = bins_slider.value,
        show_qq = qq_check.value,
        a = None, scale = scale_slider.value, loc = loc_slider.value,
        p = p_slider.value, lam = lam_slider.value,
        mu1 = mu1_slider.value, mu2 = mu2_slider.value,
        sigma1 = sigma1_slider.value, sigma2 = sigma2_slider.value,
        mix = mix_slider.value, a_pareto = a_pareto_slider.value,
        b_uniform = b_uniform_slider.value
    )


Output()